In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
import pandas as pd

In [24]:
df = pd.read_csv('sample2.csv')
df["Image_Name"] = df["Image_Name"].astype(str) + ".jpg"

In [25]:
all_ingredients = set()

for ingredients in df['Cleaned_Ingredients']:
    ingredient_list = ingredients.split(',')
    all_ingredients.update(ingredient.strip() for ingredient in ingredient_list)

all_ingredients = sorted(list(all_ingredients))

ingredient_to_idx = {ingredient: idx for idx, ingredient in enumerate(all_ingredients)}

print(ingredient_to_idx)
num_ingredients = len(all_ingredients)

{'"6 store-bought turkey meatballs (such as Murrays or Trader Joes)"': 0, "'": 1, '\'"*Chihuahua\'': 2, '\'"1 1/2 cups citrus vodka (such as Hangar One Buddhas Hand Citron or Absolut Citron)"': 3, '\'"1 1/2 cups confectioners sugar"\'': 4, '\'"1 1/2 teaspoons hawayei (see Cooks Note)': 5, '\'"1 1/3 cup confectioners sugar': 6, '\'"1 3/4 cups confectioners sugar"\']': 7, '\'"1 cup coarse fresh breadcrumbs (coarsely ground from a baguette; see Cooks Notes)"': 8, '\'"1 cup sifted confectioners sugar"': 9, '\'"1 pound (2 cups) fresh sheeps- or cows-milk ricotta': 10, '\'"1 tablespoon confectioners sugar"': 11, '\'"1 tablespoon granulated maple sugar (see cooks note': 12, '\'"1 tablespoon grated Meyer lemon zest plus 3 tablespoons Meyer lemon juice (see Cooks note': 13, '\'"1 tablespoon liquid smoke\'': 14, '\'"1 to 2 tablespoons chopped serrano or jalapeño chiles': 15, '\'"1/2 cup buffalo wing sauce (such as Franks)': 16, '\'"1/2 cup fresh ginger juice (from about 160 grams peeled fresh gi

In [26]:
import os
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications.vgg16 import preprocess_input

from google.colab import drive
drive.mount('/content/drive/')

image_folder = '/content/drive/My Drive/Sample Images2/'
image_size = (224, 224)

X = []
y = []

for idx, row in df.iterrows():
    filename = row['Image_Name']

    img_path = os.path.join(image_folder, filename)

    if os.path.exists(img_path):
        # Load and preprocess image
        img = load_img(img_path, target_size=image_size)
        img_array = img_to_array(img)
        img_array = preprocess_input(img_array)
        X.append(img_array)

        # Multi-label encoding
        labels = np.zeros(num_ingredients)
        ingredients_list = row['Cleaned_Ingredients'].split(',')
        for ing in ingredients_list:
            ing = ing.strip()
            if ing in ingredient_to_idx:
                labels[ingredient_to_idx[ing]] = 1
        y.append(labels)

X = np.array(X)
y = np.array(y)

print(X.shape, y.shape)

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).
(1000, 224, 224, 3) (1000, 9025)


In [5]:
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Freeze all layers in VGG16
for layer in base_model.layers:
    layer.trainable = False

# Add custom top layers
model = models.Sequential([
    base_model,
    layers.Flatten(),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(num_ingredients, activation='sigmoid')
])

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Show model summary
model.summary()

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ vgg16 (Functional)              │ (None, 7, 7, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │    12,845,568 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 9025)           │     4,629,825 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 32,190,081 (122.80 MB)

 Trainable params: 17,475,393 (66.66 MB)

 Non-trainable params: 14,714,688 (56.13 MB)

In [28]:
model.fit(X, y, epochs=10, batch_size=32)

Epoch 1/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 6s 171ms/step - accuracy: 0.0057 - loss: 0.0204
Epoch 2/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 10s 176ms/step - accuracy: 0.0000e+00 - loss: 0.0253
Epoch 3/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 10s 173ms/step - accuracy: 0.0000e+00 - loss: 0.0220
Epoch 4/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 10s 168ms/step - accuracy: 1.2272e-04 - loss: 0.0207
Epoch 5/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 10s 165ms/step - accuracy: 7.9515e-04 - loss: 0.0276
Epoch 6/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 10s 165ms/step - accuracy: 0.0000e+00 - loss: 0.0264
Epoch 7/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 5s 166ms/step - accuracy: 0.0000e+00 - loss: 0.0243
Epoch 8/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 5s 169ms/step - accuracy: 0.0012 - loss: 0.0244
Epoch 9/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 10s 169ms/step - accuracy: 0.0000e+00 - loss: 0.0231
Epoch 10/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 5s 168ms/step - accuracy: 0.0023 - loss: 0.0228


In [31]:
from tensorflow.keras.preprocessing import image

def predict_ingredients(img_path, model, all_ingredients, threshold=0.005):
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = preprocess_input(img_array)

    predictions = model.predict(img_array)
    print(predictions)

    predicted_labels = [
        all_ingredients[i]
        for i in range(len(all_ingredients))
        if predictions[0][i] >= threshold
    ]

    return predicted_labels

# Example usage
img_path = 'spaghetti.jpg'
ingredients = predict_ingredients(img_path, model, all_ingredients)
print('Predicted ingredients:', ingredients)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
[[7.0794440e-11 4.4071781e-11 7.9035709e-14 ... 1.0870998e-11
  8.7403253e-12 2.2022360e-10]]
Predicted ingredients: ["'1/2 teaspoon salt'", "'1/4 teaspoon salt'", "'2 garlic cloves", "'2 tablespoons olive oil'", "'Kosher salt'", "chopped'", "diced'", "divided'", "drained'", "finely chopped'", "halved'", "minced'", 'peeled', "plus more'", "thinly sliced'"]


In [33]:
model.save('vgg16_mercachef_model.keras')

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Configurar WebDriver
driver = webdriver.Chrome()

# Abrir la página
driver.get("https://tienda.mercadona.es/")

# Rechazar cookies
cookies = WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.XPATH, "//button[text()='Rechazar']"))
)
cookies.click()

# Introducir código postal
cod_post = driver.find_element(By.CSS_SELECTOR, 'input[data-testid="postal-code-checker-input"]')
cod_post.send_keys('46940')

# Aceptar código postal
aceptar = driver.find_element(By.XPATH, "//span[text()='Continuar']")
aceptar.click()

idioma = driver.find_element(By.TAG_NAME, "input")

WebDriverWait(driver, 5).until(EC.presence_of_element_located( # espera que aparezca un boton en la pantalla
    (By.CSS_SELECTOR, "a[href='/categories']")))
search = driver.find_element(By.TAG_NAME, "input")
for product in ingredients:
  search.send_keys(product)
  button = driver.find_element(By.TAG_NAME, "button")
  button.click()

driver.quit()